# 🤖 Modeling Template
Notebook ini digunakan untuk membangun dan mengevaluasi model Machine Learning/Data Mining.

## Langkah yang disarankan:
- Pisahkan data latih dan data uji
- Latih model (misalnya: Decision Tree, Random Forest, SVM, dll.)
- Evaluasi menggunakan metrik: accuracy, precision, recall, F1-score, ROC, confusion matrix

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit

from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Library berhasil diimport")

In [ ]:
# LOAD DATASET


DATA_PATH = "../data/preprocessed/data_preprocessing_encoded.csv"

df_model_encoded = pd.read_csv(DATA_PATH)

df_model_encoded.head()

###Dynamic Time-Based Split

Split dibuat dinamis agar saat data bertambah, pembagian data tetap menyesuaikan.

In [ ]:
df_model_encoded = df_model_encoded.sort_values("Tanggal").reset_index(drop=True)

# Ambil tanggal unik
unique_dates = sorted(df_model_encoded["Tanggal"].unique())

# Tentukan titik split 80% berdasarkan tanggal unik
split_index = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_index]

train_data = df_model_encoded[df_model_encoded["Tanggal"] < split_date]
test_data = df_model_encoded[df_model_encoded["Tanggal"] >= split_date]

X_train = train_data[fitur_final]
y_train = train_data[target]

X_test = test_data[fitur_final]
y_test = test_data[target]

print("Split date:", split_date)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nPeriode train:")
print(train_data["Tanggal"].min(), "sampai", train_data["Tanggal"].max())

print("\nPeriode test:")
print(test_data["Tanggal"].min(), "sampai", test_data["Tanggal"].max())

###Scaling Fitur Numerik

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[fitur_numerik] = scaler.fit_transform(X_train[fitur_numerik])
X_test_scaled[fitur_numerik] = scaler.transform(X_test[fitur_numerik])

display(X_train_scaled.head())

###Fungsi Evaluasi

In [ ]:
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2
    }

In [ ]:
def print_header(title):
    print("=" * 70)
    print(title)
    print("=" * 70)
    print()

In [ ]:
def run_single_model_cv_tuning(
    model_name,
    tuning_name,
    search_class,
    estimator,
    param_space,
    X_train,
    y_train,
    X_test,
    y_test,
    cv,
    use_scaler=False,
    n_iter=None,
    random_state=42
):
    print_header(f"CROSS VALIDATION: {model_name} | TUNING: {tuning_name}")

    if use_scaler:
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", estimator)
        ])
    else:
        pipeline = Pipeline([
            ("scaler", "passthrough"),
            ("model", estimator)
        ])

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train), 1):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        if search_class == GridSearchCV:
            search = search_class(
                estimator=pipeline,
                param_grid=param_space,
                cv=3,
                scoring="neg_root_mean_squared_error",
                n_jobs=-1
            )

        elif search_class == RandomizedSearchCV:
            search = search_class(
                estimator=pipeline,
                param_distributions=param_space,
                n_iter=n_iter,
                cv=3,
                scoring="neg_root_mean_squared_error",
                random_state=random_state,
                n_jobs=-1
            )

        elif search_class == BayesSearchCV:
            search = search_class(
                estimator=pipeline,
                search_spaces=param_space,
                n_iter=n_iter,
                cv=3,
                scoring="neg_root_mean_squared_error",
                random_state=random_state,
                n_jobs=-1
            )

        search.fit(X_fold_train, y_fold_train)

        best_model = search.best_estimator_
        y_fold_pred = best_model.predict(X_fold_val)

        metrics = calculate_metrics(y_fold_val, y_fold_pred)

        fold_results.append({
            "Model": model_name,
            "Tuning": tuning_name,
            "Fold": fold,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
            "R2": metrics["R2"],
            "Best_Params": search.best_params_
        })

        print(f"Fold {fold}/{cv.n_splits}")
        print(
            f"Best Fold MAE: {metrics['MAE']:.4f} | "
            f"RMSE: {metrics['RMSE']:.4f} | "
            f"MAPE: {metrics['MAPE']:.4f}% | "
            f"R2: {metrics['R2']:.4f}"
        )
        print("Best Params:", search.best_params_)
        print()

    fold_results_df = pd.DataFrame(fold_results)

    print_header(f"FINAL RESULT: {model_name} | TUNING: {tuning_name}")

    print(f"MAE  = {fold_results_df['MAE'].mean():.3f} ± {fold_results_df['MAE'].std():.3f}")
    print(f"RMSE = {fold_results_df['RMSE'].mean():.3f} ± {fold_results_df['RMSE'].std():.3f}")
    print(f"MAPE = {fold_results_df['MAPE'].mean():.3f} ± {fold_results_df['MAPE'].std():.3f}%")
    print(f"R²   = {fold_results_df['R2'].mean():.3f} ± {fold_results_df['R2'].std():.3f}")
    print()

    print_header(f"FINAL TRAINING ON FULL TRAIN DATA: {model_name} | TUNING: {tuning_name}")

    if search_class == GridSearchCV:
        final_search = search_class(
            estimator=pipeline,
            param_grid=param_space,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

    elif search_class == RandomizedSearchCV:
        final_search = search_class(
            estimator=pipeline,
            param_distributions=param_space,
            n_iter=n_iter,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            random_state=random_state,
            n_jobs=-1
        )

    elif search_class == BayesSearchCV:
        final_search = search_class(
            estimator=pipeline,
            search_spaces=param_space,
            n_iter=n_iter,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            random_state=random_state,
            n_jobs=-1
        )

    final_search.fit(X_train, y_train)

    final_best_model = final_search.best_estimator_
    y_test_pred = final_best_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    print("Best Parameters:")
    print(final_search.best_params_)
    print()
    print(
        f"TEST RESULT | "
        f"MAE: {test_metrics['MAE']:.4f} | "
        f"RMSE: {test_metrics['RMSE']:.4f} | "
        f"MAPE: {test_metrics['MAPE']:.4f}% | "
        f"R2: {test_metrics['R2']:.4f}"
    )
    print()

    final_result = {
        "Model": model_name,
        "Tuning": tuning_name,
        "Test_MAE": test_metrics["MAE"],
        "Test_RMSE": test_metrics["RMSE"],
        "Test_MAPE": test_metrics["MAPE"],
        "Test_R2": test_metrics["R2"],
        "Best_Params": final_search.best_params_,
        "Best_Estimator": final_best_model,
        "Prediction": y_test_pred
    }

    return fold_results_df, final_result

In [ ]:
pipeline = Pipeline([
    ("scaler", "passthrough"),
    ("model", LinearRegression())
])

In [ ]:
#cross validation time series
tscv = TimeSeriesSplit(n_splits=5)

###Parameter Tuning

In [ ]:
param_lr = {
    "model__fit_intercept": [True, False],
    "model__positive": [False, True]
}


In [ ]:
param_rf = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

In [ ]:
###Training Linear Regression with GridSearchCV

In [ ]:
fold_lr_grid, hasil_lr_grid = run_single_model_cv_tuning(
    model_name="Linear Regression",
    tuning_name="Grid Search",
    search_class=GridSearchCV,
    estimator=LinearRegression(),
    param_space=param_lr,
    X_train=X_train_scaled,
    y_train=y_train,
    X_test=X_test_scaled,
    y_test=y_test,
    cv=tscv,
    use_scaler=True
)

In [ ]:
####Training Random Forest with GridSearchCV

In [ ]:
fold_rf_grid, hasil_rf_grid = run_single_model_cv_tuning(
    model_name="Random Forest Regression",
    tuning_name="Grid Search",
    search_class=GridSearchCV,
    estimator=RandomForestRegressor(random_state=42),
    param_space=param_rf,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    cv=tscv,
    use_scaler=False
)

###Training Linear Regression with RandomSearch

In [ ]:
fold_lr_random, hasil_lr_random = run_single_model_cv_tuning(
    model_name="Linear Regression",
    tuning_name="Random Search",
    search_class=RandomizedSearchCV,
    estimator=LinearRegression(),
    param_space=param_lr,
    X_train=X_train_scaled,
    y_train=y_train,
    X_test=X_test_scaled,
    y_test=y_test,
    cv=tscv,
    use_scaler=True,
    n_iter=4,
    random_state=42
)

###Training Random Forest with RandomSearch

In [ ]:
fold_rf_random, hasil_rf_random = run_single_model_cv_tuning(
    model_name="Random Forest Regression",
    tuning_name="Random Search",
    search_class=RandomizedSearchCV,
    estimator=RandomForestRegressor(random_state=42),
    param_space=param_rf,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    cv=tscv,
    use_scaler=False,
    n_iter=30,
    random_state=42
)

###Training Linear Regression with Bayesian

In [ ]:
fold_lr_bayes, hasil_lr_bayes = run_single_model_cv_tuning(
    model_name="Linear Regression",
    tuning_name="Bayesian Optimization",
    search_class=BayesSearchCV,
    estimator=LinearRegression(),
    param_space=param_lr,
    X_train=X_train_scaled,
    y_train=y_train,
    X_test=X_test_scaled,
    y_test=y_test,
    cv=tscv,
    use_scaler=True,
    n_iter=10,
    random_state=42
)

###Training Random Forest with Bayesian

In [ ]:
fold_rf_bayes, hasil_rf_bayes = run_single_model_cv_tuning(
    model_name="Random Forest Regression",
    tuning_name="Bayesian Optimization",
    search_class=BayesSearchCV,
    estimator=RandomForestRegressor(random_state=42),
    param_space=param_rf,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    cv=tscv,
    use_scaler=False,
    n_iter=30,
    random_state=42
)

###Evaluasi

In [ ]:
hasil_semua_model = pd.DataFrame([
    {
        "Model": hasil_lr_grid["Model"],
        "Tuning": hasil_lr_grid["Tuning"],
        "MAE": hasil_lr_grid["Test_MAE"],
        "RMSE": hasil_lr_grid["Test_RMSE"],
        "MAPE": hasil_lr_grid["Test_MAPE"],
        "R2": hasil_lr_grid["Test_R2"]
    },
    {
        "Model": hasil_lr_random["Model"],
        "Tuning": hasil_lr_random["Tuning"],
        "MAE": hasil_lr_random["Test_MAE"],
        "RMSE": hasil_lr_random["Test_RMSE"],
        "MAPE": hasil_lr_random["Test_MAPE"],
        "R2": hasil_lr_random["Test_R2"]
    },
    {
        "Model": hasil_lr_bayes["Model"],
        "Tuning": hasil_lr_bayes["Tuning"],
        "MAE": hasil_lr_bayes["Test_MAE"],
        "RMSE": hasil_lr_bayes["Test_RMSE"],
        "MAPE": hasil_lr_bayes["Test_MAPE"],
        "R2": hasil_lr_bayes["Test_R2"]
    },
    {
        "Model": hasil_rf_grid["Model"],
        "Tuning": hasil_rf_grid["Tuning"],
        "MAE": hasil_rf_grid["Test_MAE"],
        "RMSE": hasil_rf_grid["Test_RMSE"],
        "MAPE": hasil_rf_grid["Test_MAPE"],
        "R2": hasil_rf_grid["Test_R2"]
    },
    {
        "Model": hasil_rf_random["Model"],
        "Tuning": hasil_rf_random["Tuning"],
        "MAE": hasil_rf_random["Test_MAE"],
        "RMSE": hasil_rf_random["Test_RMSE"],
        "MAPE": hasil_rf_random["Test_MAPE"],
        "R2": hasil_rf_random["Test_R2"]
    },
    {
        "Model": hasil_rf_bayes["Model"],
        "Tuning": hasil_rf_bayes["Tuning"],
        "MAE": hasil_rf_bayes["Test_MAE"],
        "RMSE": hasil_rf_bayes["Test_RMSE"],
        "MAPE": hasil_rf_bayes["Test_MAPE"],
        "R2": hasil_rf_bayes["Test_R2"]
    }
])

hasil_semua_model = hasil_semua_model.sort_values("RMSE", ascending=True)

display(hasil_semua_model)

In [ ]:
best_row = hasil_semua_model.iloc[0]

print_header("BEST OVERALL MODEL")

print("Best Model :", best_row["Model"])
print("Best Tuning:", best_row["Tuning"])
print(f"MAE        : {best_row['MAE']:.4f}")
print(f"RMSE       : {best_row['RMSE']:.4f}")
print(f"MAPE       : {best_row['MAPE']:.4f}%")
print(f"R2         : {best_row['R2']:.4f}")

In [ ]:
if best_row["Model"] == "Linear Regression" and best_row["Tuning"] == "Grid Search":
    model_terbaik = hasil_lr_grid["Best_Estimator"]
    y_pred_terbaik = hasil_lr_grid["Prediction"]

elif best_row["Model"] == "Linear Regression" and best_row["Tuning"] == "Random Search":
    model_terbaik = hasil_lr_random["Best_Estimator"]
    y_pred_terbaik = hasil_lr_random["Prediction"]

elif best_row["Model"] == "Linear Regression" and best_row["Tuning"] == "Bayesian Optimization":
    model_terbaik = hasil_lr_bayes["Best_Estimator"]
    y_pred_terbaik = hasil_lr_bayes["Prediction"]

elif best_row["Model"] == "Random Forest Regression" and best_row["Tuning"] == "Grid Search":
    model_terbaik = hasil_rf_grid["Best_Estimator"]
    y_pred_terbaik = hasil_rf_grid["Prediction"]

elif best_row["Model"] == "Random Forest Regression" and best_row["Tuning"] == "Random Search":
    model_terbaik = hasil_rf_random["Best_Estimator"]
    y_pred_terbaik = hasil_rf_random["Prediction"]

else:
    model_terbaik = hasil_rf_bayes["Best_Estimator"]
    y_pred_terbaik = hasil_rf_bayes["Prediction"]

print("Model terbaik berhasil dipilih.")

In [ ]:
###Visualisasi Perbandingan Hasil

In [ ]:
hasil_semua_model["Model_Tuning"] = (
    hasil_semua_model["Model"] + " - " + hasil_semua_model["Tuning"]
)

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(hasil_semua_model["Model_Tuning"], hasil_semua_model["RMSE"])
plt.title("Perbandingan RMSE Semua Model")
plt.xlabel("Model dan Tuning")
plt.ylabel("RMSE")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(hasil_semua_model["Model_Tuning"], hasil_semua_model["MAE"])
plt.title("Perbandingan MAE Semua Model")
plt.xlabel("Model dan Tuning")
plt.ylabel("MAE")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(hasil_semua_model["Model_Tuning"], hasil_semua_model["MAPE"])
plt.title("Perbandingan MAPE Semua Model")
plt.xlabel("Model dan Tuning")
plt.ylabel("MAPE (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(hasil_semua_model["Model_Tuning"], hasil_semua_model["R2"])
plt.title("Perbandingan R² Semua Model")
plt.xlabel("Model dan Tuning")
plt.ylabel("R² Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

###Tabel Prediksi Aktual vs Prediksi

In [ ]:
hasil_prediksi = test_data.copy()

hasil_prediksi["Harga_Aktual"] = y_test.values
hasil_prediksi["Harga_Prediksi"] = y_pred_terbaik
hasil_prediksi["Absolute_Error"] = abs(
    hasil_prediksi["Harga_Aktual"] - hasil_prediksi["Harga_Prediksi"]
)

display(
    hasil_prediksi[
        [
            "Tanggal",
            "Tahun",
            "Bulan",
            "Harga",
            "Harga_Aktual",
            "Harga_Prediksi",
            "Absolute_Error"
        ]
    ].head(20)
)

plt.figure(figsize=(14, 6))

plt.plot(
    hasil_prediksi["Tanggal"],
    hasil_prediksi["Harga_Aktual"],
    marker="o",
    label="Harga Aktual"
)

plt.plot(
    hasil_prediksi["Tanggal"],
    hasil_prediksi["Harga_Prediksi"],
    marker="x",
    label="Harga Prediksi"
)

plt.title("Perbandingan Harga Aktual dan Prediksi Model Terbaik")
plt.xlabel("Tanggal")
plt.ylabel("Harga")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()